# 🍽️ Analisis Sentimen Komentar Program MBG
## Perbandingan Naïve Bayes vs Logistic Regression
---
**Mata Kuliah:** Pembelajaran Mesin — Pertemuan 7  
**Dataset:** 6.419 Komentar Twitter Program Makan Bergizi Gratis (MBG)  
**Periode:** Januari – Oktober 2025  
**Metode:** Multinomial Naïve Bayes & Logistic Regression  
**Novelty:** Komparasi dua algoritma + 3 kelas sentimen vs jurnal pembanding (NB + 2 kelas)

## 📦 1. Import Library & Install Dependensi

In [ ]:
# Install PySastrawi untuk stemming bahasa Indonesia
# Jalankan sekali, kemudian bisa dikomentari
!pip install PySastrawi -q
!pip install joblib -q

import pandas as pd
import numpy as np
import re
import html
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score)
import joblib

# Import PySastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

print('✅ Semua library berhasil diimport!')
print(f'   PySastrawi stemmer siap digunakan')

## 📂 2. Load Dataset

In [ ]:
df = pd.read_csv('Data_Sentimen_MBG.csv')
print(f'Total data awal: {len(df)}')
print(f'Kolom: {df.columns.tolist()}')
print()
df.head()

## 🧹 3. Text Preprocessing

### 3.1 Cleaning Teks

Membersihkan teks dari noise: HTML entities, mention, URL, hashtag, karakter khusus, dan angka.

**Fix:** `html.unescape()` di awal agar `&quot;` → `"` dan `&amp;` → `&` di-decode **sebelum** regex berjalan.

In [ ]:
df = df[['text']].dropna()
df['text'] = df['text'].astype(str)

def clean_text(text):
    # STEP 1: Decode HTML entities DULU
    text = html.unescape(text)
    # STEP 2: Hapus sisa entity HTML
    text = re.sub(r'&\w+;', ' ', text)
    # Hapus mention (@username)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus simbol hashtag, simpan kata
    text = re.sub(r'#(\w+)', r'\1', text)
    # Hapus karakter khusus & emoji
    text = re.sub(r'[^\w\s]', ' ', text)
    # Hapus angka
    text = re.sub(r'\d+', '', text)
    # Lowercase
    text = text.lower()
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned'] = df['text'].apply(clean_text)
print('✅ Cleaning selesai!')
df[['text','cleaned']].head(5)

### 3.2 Stopword Removal

In [ ]:
stopwords_id = set([
    # Kata umum
    'yang','dan','di','dengan','untuk','tidak','ini','dari','dalam','akan',
    'pada','juga','saya','ke','karena','tersebut','bisa','ada','mereka','lebih',
    'kata','tahun','sudah','atau','saat','oleh','menjadi','orang','itu','kita',
    'kami','kalau','kalian','kamu','dia','apa','juga','tapi','tp','ya','yg',
    'ny','nya','dgn','utk','jg','sy','gak','ga','aja','udah','dah','nih','sih',
    'lah','kan','dong','lagi','buat','lg','dr','krn','sdh','pd','klo','kl',
    'emg','emang','memang','banget','bgt','mau','gimana','gmn','kayak','kaya',
    'gitu','gini','kok','deh','wkwk','wkwkwk','haha','hehe','eh','ah','oh',
    'iya','ok','oke','yuk','ajah','adalah','bahwa','hal','bagi','serta','jika',
    'bila','hingga','antara','namun','maka','sejak','setelah','sebelum','sebab',
    # Singkatan
    'tdk','gak','ga','ny','tp','yg','jg','sy','bgt','krn','sdh','klo','kl',
    'lg','dr','pd','dgn','utk','dll','dsb','dst',
    # Sisa artefak HTML
    'quot','amp','apos','nbsp','lt','gt',
    # Sapaan
    'min','kak','pak','bu','mas','mbak','bang','bro','sis','sob','guys',
    # Penghubung
    'namun','tetapi','meski','meskipun','walaupun','agar','supaya','bahkan',
    'padahal','sedangkan','sehingga','kemudian','lalu','selanjutnya','akhirnya',
    # Tanya & partikel
    'siapa','dimana','kemana','darimana','kapan','bagaimana','kenapa','mengapa',
    'pun','pula','jua','kah','tah','nah','neh',
    # Nama program MBG
    'mbg','makan','bergizi','gratis',
    # Nama tokoh
    'prabowo','jokowi','joko','widodo','gibran','megawati',
    'anies','ganjar','ahok','sby','yudhoyono',
    # Nama tempat
    'indonesia','jakarta','jawa','surabaya','bandung','medan',
    'bali','sulawesi','kalimantan','papua','sumatra',
    # Portal berita
    'tempo','kompas','detik','tribun','cnbc','okezone',
    'liputan','republika','kumparan','antara',
    # Kata konteks umum tidak diskriminatif
    'negara','rakyat','program','proyek','pemerintah',
    'presiden','menteri','anak','sekolah','siswa','murid',
    'daerah','dinas','kabupaten','kota','provinsi',
    'anggaran','dana','uang','biaya','rupiah',
])

def remove_stopwords(text):
    words = text.split()
    return ' '.join([w for w in words
                     if w not in stopwords_id and len(w) > 2])

df['no_stopwords'] = df['cleaned'].apply(remove_stopwords)
print('✅ Stopword removal selesai!')
df[['text','cleaned','no_stopwords']].head(5)

### 3.3 Stemming dengan PySastrawi ✨ (Baru)

Stemming menggabungkan variasi kata bermakna sama menjadi satu bentuk dasar.

Contoh: `korupsi`, `korup`, `dikorupsi`, `mengkorupsi` → **`korup`**

Manfaat: membuat model lebih efisien karena fitur yang mirip digabungkan.

In [ ]:
def apply_stemming(text):
    """Terapkan stemming Sastrawi pada setiap kata dalam teks."""
    words = text.split()
    stemmed = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed)

print('⏳ Proses stemming... (membutuhkan beberapa menit untuk dataset besar)')
df['processed'] = df['no_stopwords'].apply(apply_stemming)
print('✅ Stemming selesai!')

# Tampilkan contoh before/after stemming
print('\n' + '='*60)
print('CONTOH BEFORE vs AFTER STEMMING')
print('='*60)
contoh_kata = [
    'korupsi dikorupsi mengkorupsi koruptor',
    'membantu terbantu pembantu bantuan',
    'kecewa mengecewakan kekecewaan',
    'keracunan meracuni beracun racun',
    'manfaat bermanfaat memanfaatkan'
]
for kalimat in contoh_kata:
    hasil = apply_stemming(kalimat)
    print(f'Sebelum : {kalimat}')
    print(f'Sesudah : {hasil}')
    print()

# Tampilkan perbandingan pada data asli
print('='*60)
print('CONTOH PADA DATA ASLI')
print('='*60)
df[['text','no_stopwords','processed']].head(5)

## 🏷️ 4. Auto Labeling Sentimen (Lexicon-Based)

**Perbaikan kamus lexicon:**
- ❌ **Dihapus** kata ambigu: `gratis`, `gizi`, `pekerjaan`, `kerja`, `rezeki`, `benar`, `tepat` → kata-kata ini bermakna faktual, bukan ekspresi sentimen
- ✅ **Ditambah** kata negatif yang lebih kuat dan spesifik emosinya
- ✅ **Ditambah** kata positif yang benar-benar ekspresi sentimen subjektif

In [ ]:
# ============================================================
# KAMUS LEXICON — DIPERBAIKI
# ============================================================

kata_positif = [
    # Ekspresi positif umum
    'bagus','baik','setuju','dukung','mendukung','bermanfaat','manfaat',
    'alhamdulillah','sukses','berhasil','senang','bangga','hebat','keren',
    'mantap','sehat','nutrisi','harapan','positif','maju','sejahtera',
    'membantu','terbantu','solusi','nikmat','enak','lezat',
    'bermanfaat','berguna','efektif','transparan','akuntabel',
    'merata','adil','tepat sasaran','berkualitas',
    # ✅ Tambahan — ekspresi sentimen positif yang lebih kuat
    'luar biasa','mengagumkan','memukau','inspiratif','membanggakan',
    'berhasil','sukses','berhasil guna','berdaya guna',
    'apresiasi','salut','respect','kagum','syukur','bersyukur',
    'semangat','antusias','optimis','harapan','percaya',
    'lancar','tertib','rapi','teratur','disiplin',
    'inovatif','kreatif','progresif','produktif',
    'puas','memuaskan','memuaskan','memuaskan',
    'suka','senang hati','dengan senang','gembira','bahagia','senyum',
    'terima kasih','makasih','trimakasih','terimakasih','terharu',
    # ❌ DIHAPUS dari positif (kata ambigu / faktual):
    # 'gratis' → fakta program, bukan sentimen
    # 'gizi' → istilah teknis, bukan sentimen
    # 'pekerjaan','kerja','rezeki' → fakta, muncul di semua kelas
    # 'benar','tepat' → kata penilaian netral, terlalu ambigu
]

kata_negatif = [
    # Kata negatif umum
    'korupsi','korup','maling','bancakan','rusak','gagal','bohong','palsu',
    'penipuan','tipu','biadab','bangsat','hancur','ancur','buruk','jelek',
    'sampah','bubar','tolol','bodoh','goblok','idiot','brengsek',
    'racun','bahaya','berbahaya','keracunan','mati','kematian','beracun',
    'kecewa','marah','protes','demo','tolak','menolak','sia sia','percuma',
    'pemborosan','boros','hutang','utang','koruptor','nepotisme','menipu',
    'rezim','otoriter','amburadul','kacau','berantakan','malakin','markup',
    'stop','bubarkan','hentikan','tidak layak','tidak berguna','tidak bener',
    'tidak benar','gak bener','salah','merugikan','dikorupsi','makan racun',
    # ✅ Tambahan — kata negatif yang lebih spesifik dan kuat
    'gagal total','total gagal','tidak berhasil','tidak efektif',
    'menghancurkan','merusak','membahayakan','mengancam',
    'tidak merata','tidak adil','diskriminatif','pilih kasih',
    'tidak transparan','gelap','ditilep','dimark up','diselewengkan',
    'basi','busuk','tidak layak makan','tidak higienis','kotor',
    'mual','muntah','sakit','kesakitan','tergangu','terganggu',
    'mahal','kemahalan','tidak terjangkau','memberatkan',
    'lambat','terlambat','molor','mangkrak','terbengkalai',
    'dipotong','sunat','disunat','pemotongan','pengurangan paksa',
    'formalitas','pencitraan','hanya pencitraan','gimmick','lips service',
    'tidak serius','setengah hati','abal abal','bohong belaka',
    'menyedihkan','ironis','miris','memprihatinkan','mengkhawatirkan',
    'benci','muak','jijik','melecehkan','menghina','mempermalukan',
]

print(f'✅ Kamus positif: {len(kata_positif)} kata')
print(f'✅ Kamus negatif: {len(kata_negatif)} kata')
print()
print('📌 Kata yang DIHAPUS dari kamus (terlalu ambigu):')
print('   Positif: gratis, gizi, pekerjaan, kerja, rezeki, benar, tepat')
print('   (Kata-kata ini muncul di semua kelas, bukan ekspresi sentimen murni)')

### 4.1 Fungsi Labeling + Confidence Score ✨ (Baru)

**Confidence Score** mengukur seberapa yakin lexicon dalam menentukan label.

Formula: `confidence = abs(skor_pos - skor_neg) / (total_kata_sentimen + 1)`

- Confidence tinggi (> 0.6) → label lebih dapat dipercaya
- Confidence rendah (< 0.3) → label kurang pasti, perlu validasi manual

In [ ]:
def label_sentimen_dengan_confidence(text):
    """Label sentimen + hitung confidence score."""
    words = text.lower()
    
    # Hitung skor positif dan negatif
    pos = sum(1 for k in kata_positif if k in words)
    neg = sum(1 for k in kata_negatif if k in words)
    
    # Tentukan label
    if neg > pos:
        label = 'negatif'
    elif pos > neg:
        label = 'positif'
    else:
        label = 'netral'
    
    # Hitung confidence score
    total_sentimen = pos + neg
    confidence = abs(pos - neg) / (total_sentimen + 1)
    
    return label, confidence, pos, neg

# Terapkan ke seluruh data
hasil = df['processed'].apply(label_sentimen_dengan_confidence)
df['sentimen']    = hasil.apply(lambda x: x[0])
df['confidence']  = hasil.apply(lambda x: x[1])
df['skor_pos']    = hasil.apply(lambda x: x[2])
df['skor_neg']    = hasil.apply(lambda x: x[3])

# Distribusi sentimen
dist = df['sentimen'].value_counts()
print('📊 Distribusi Sentimen (Auto Label):')
for s, c in dist.items():
    print(f'  {s:10s}: {c:4d} komentar ({c/len(df)*100:.1f}%)')
print(f'\n  Total    : {len(df)} komentar')

# Statistik confidence
print('\n📊 Statistik Confidence Score:')
print(f'  Rata-rata  : {df["confidence"].mean():.4f}')
print(f'  Minimum    : {df["confidence"].min():.4f}')
print(f'  Maksimum   : {df["confidence"].max():.4f}')

# Analisis distribusi confidence
rendah   = (df['confidence'] < 0.3).sum()
sedang   = ((df['confidence'] >= 0.3) & (df['confidence'] <= 0.6)).sum()
tinggi   = (df['confidence'] > 0.6).sum()
total    = len(df)
print(f'\n📊 Distribusi Confidence Score:')
print(f'  Rendah  (< 0.3) : {rendah:4d} data ({rendah/total*100:.1f}%) — perlu validasi manual')
print(f'  Sedang (0.3–0.6): {sedang:4d} data ({sedang/total*100:.1f}%)')
print(f'  Tinggi (> 0.6)  : {tinggi:4d} data ({tinggi/total*100:.1f}%) — label paling reliabel')

### 4.2 Visualisasi Distribusi Confidence Score ✨ (Baru)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle('Distribusi Confidence Score Labeling Lexicon per Kelas Sentimen',
             fontsize=14, fontweight='bold', y=1.02)

colors_conf = {'negatif': '#e74c3c', 'netral': '#95a5a6', 'positif': '#2ecc71'}
sentimen_labels = ['negatif', 'netral', 'positif']

for i, sentimen in enumerate(sentimen_labels):
    ax = axes[i]
    data_conf = df[df['sentimen'] == sentimen]['confidence']
    
    # Histogram
    n, bins, patches = ax.hist(data_conf, bins=20, color=colors_conf[sentimen],
                                edgecolor='white', linewidth=0.8, alpha=0.85)
    
    # Garis vertikal threshold
    ax.axvline(x=0.3, color='orange', linestyle='--', linewidth=1.5, label='Batas rendah (0.3)')
    ax.axvline(x=0.6, color='blue',   linestyle='--', linewidth=1.5, label='Batas tinggi (0.6)')
    ax.axvline(x=data_conf.mean(), color='black', linestyle='-', linewidth=2,
               label=f'Rata-rata ({data_conf.mean():.2f})')
    
    # Statistik
    r = (data_conf < 0.3).sum()
    s = ((data_conf >= 0.3) & (data_conf <= 0.6)).sum()
    t = (data_conf > 0.6).sum()
    total_s = len(data_conf)
    
    ax.set_title(f'{sentimen.upper()}\n({total_s} data)', fontweight='bold', fontsize=12)
    ax.set_xlabel('Confidence Score')
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_facecolor('#f8f9fa')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Anotasi distribusi
    textstr = f'Rendah: {r} ({r/total_s*100:.0f}%)\nSedang: {s} ({s/total_s*100:.0f}%)\nTinggi: {t} ({t/total_s*100:.0f}%)'
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('confidence_score_distribution.png', dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print('✅ Grafik confidence score disimpan: confidence_score_distribution.png')

## ✂️ 5. Split Data & TF-IDF Vectorizer

In [ ]:
X = df['processed']
y = df['sentimen']

# Split 80:20 dengan stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'📁 Data Training : {len(X_train)} data ({len(X_train)/len(X)*100:.0f}%)')
print(f'📁 Data Testing  : {len(X_test)} data ({len(X_test)/len(X)*100:.0f}%)')

## 🔬 6. Eksperimen Unigram vs Bigram ✨ (Baru)

Membandingkan 4 kombinasi:
- **NB + Unigram** (1 kata per fitur)
- **NB + Bigram** (1-2 kata per fitur)
- **LR + Unigram**
- **LR + Bigram**

In [ ]:
hasil_eksperimen = []

for ngram_label, ngram_range in [('Unigram (1,1)', (1,1)), ('Bigram (1,2)', (1,2))]:
    # Vectorizer
    tfidf_exp = TfidfVectorizer(max_features=5000, ngram_range=ngram_range)
    X_train_tf_exp = tfidf_exp.fit_transform(X_train)
    X_test_tf_exp  = tfidf_exp.transform(X_test)
    
    for model_label, model in [
        ('Naïve Bayes', MultinomialNB()),
        ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42))
    ]:
        model.fit(X_train_tf_exp, y_train)
        y_pred_exp = model.predict(X_test_tf_exp)
        acc = accuracy_score(y_test, y_pred_exp)
        f1_macro = f1_score(y_test, y_pred_exp, average='macro')
        f1_weighted = f1_score(y_test, y_pred_exp, average='weighted')
        hasil_eksperimen.append({
            'Model': model_label,
            'N-gram': ngram_label,
            'Akurasi (%)': round(acc * 100, 2),
            'F1 Macro': round(f1_macro, 4),
            'F1 Weighted': round(f1_weighted, 4)
        })
        print(f'✅ {model_label} + {ngram_label}: Akurasi = {acc*100:.2f}%')

df_eksperimen = pd.DataFrame(hasil_eksperimen)
print()
print('='*65)
print('TABEL PERBANDINGAN — UNIGRAM vs BIGRAM')
print('='*65)
print(df_eksperimen.to_string(index=False))
print()

# Identifikasi kombinasi terbaik
best_idx = df_eksperimen['Akurasi (%)'].idxmax()
best = df_eksperimen.loc[best_idx]
print(f'🏆 Kombinasi Terbaik: {best["Model"]} + {best["N-gram"]}')
print(f'   Akurasi: {best["Akurasi (%)"]:.2f}%')

### 6.1 Visualisasi Perbandingan Unigram vs Bigram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle('Eksperimen Unigram vs Bigram\nPerbandingan Akurasi & F1 Score',
             fontsize=14, fontweight='bold')

# Data untuk plot
nb_uni  = df_eksperimen[(df_eksperimen['Model']=='Naïve Bayes') & (df_eksperimen['N-gram']=='Unigram (1,1)')]['Akurasi (%)'].values[0]
nb_bi   = df_eksperimen[(df_eksperimen['Model']=='Naïve Bayes') & (df_eksperimen['N-gram']=='Bigram (1,2)')]['Akurasi (%)'].values[0]
lr_uni  = df_eksperimen[(df_eksperimen['Model']=='Logistic Regression') & (df_eksperimen['N-gram']=='Unigram (1,1)')]['Akurasi (%)'].values[0]
lr_bi   = df_eksperimen[(df_eksperimen['Model']=='Logistic Regression') & (df_eksperimen['N-gram']=='Bigram (1,2)')]['Akurasi (%)'].values[0]

# Plot 1 — Grouped Bar Chart Akurasi
ax1 = axes[0]
x = np.arange(2)
bars1 = ax1.bar(x - 0.2, [nb_uni, nb_bi], 0.35, label='Naïve Bayes', color='#3498db', edgecolor='white')
bars2 = ax1.bar(x + 0.2, [lr_uni, lr_bi], 0.35, label='Logistic Regression', color='#e67e22', edgecolor='white')
for bar in bars1:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{bar.get_height():.2f}%',
             ha='center', fontsize=10, fontweight='bold')
for bar in bars2:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{bar.get_height():.2f}%',
             ha='center', fontsize=10, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(['Unigram (1,1)', 'Bigram (1,2)'], fontsize=11)
ax1.set_ylim(0, 105)
ax1.set_ylabel('Akurasi (%)')
ax1.set_title('Perbandingan Akurasi', fontweight='bold')
ax1.legend()
ax1.set_facecolor('#f8f9fa')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Plot 2 — Tabel ringkasan
ax2 = axes[1]
ax2.axis('off')
table_data = [
    ['NB + Unigram',  f'{nb_uni:.2f}%',  df_eksperimen[(df_eksperimen['Model']=='Naïve Bayes') & (df_eksperimen['N-gram']=='Unigram (1,1)')]['F1 Macro'].values[0]],
    ['NB + Bigram',   f'{nb_bi:.2f}%',   df_eksperimen[(df_eksperimen['Model']=='Naïve Bayes') & (df_eksperimen['N-gram']=='Bigram (1,2)')]['F1 Macro'].values[0]],
    ['LR + Unigram',  f'{lr_uni:.2f}%',  df_eksperimen[(df_eksperimen['Model']=='Logistic Regression') & (df_eksperimen['N-gram']=='Unigram (1,1)')]['F1 Macro'].values[0]],
    ['LR + Bigram',   f'{lr_bi:.2f}%',   df_eksperimen[(df_eksperimen['Model']=='Logistic Regression') & (df_eksperimen['N-gram']=='Bigram (1,2)')]['F1 Macro'].values[0]],
]
col_labels = ['Kombinasi', 'Akurasi', 'F1 Macro']
table = ax2.table(cellText=table_data, colLabels=col_labels,
                  cellLoc='center', loc='center',
                  colColours=['#bdc3c7']*3)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.3, 2.0)
# Highlight baris terbaik
best_row_data = [nb_uni, nb_bi, lr_uni, lr_bi]
best_row_idx  = best_row_data.index(max(best_row_data)) + 1
for j in range(3):
    table[best_row_idx, j].set_facecolor('#f39c12')
    table[best_row_idx, j].set_text_props(fontweight='bold')
ax2.set_title('Ringkasan (🏆 = Terbaik)', fontweight='bold')

plt.tight_layout()
plt.savefig('unigram_vs_bigram.png', dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print('✅ Grafik disimpan: unigram_vs_bigram.png')

## 🤖 7. Model Final — Naïve Bayes & Logistic Regression (Bigram Terbaik)

In [ ]:
# Gunakan konfigurasi bigram (berdasarkan hasil eksperimen)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

print(f'✅ TF-IDF Matrix Shape (train): {X_train_tf.shape}')
print(f'   ({X_train_tf.shape[0]} dokumen x {X_train_tf.shape[1]} fitur)')

In [ ]:
# === Model 1: Naïve Bayes ===
nb = MultinomialNB()
nb.fit(X_train_tf, y_train)
y_pred_nb = nb.predict(X_test_tf)
acc_nb = accuracy_score(y_test, y_pred_nb)

print('='*45)
print('        HASIL EVALUASI NAÏVE BAYES')
print('='*45)
print(f'  Akurasi: {acc_nb*100:.2f}%')
print()
print(classification_report(y_test, y_pred_nb,
      target_names=['negatif','netral','positif']))

In [ ]:
# === Model 2: Logistic Regression ===
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tf, y_train)
y_pred_lr = lr.predict(X_test_tf)
acc_lr = accuracy_score(y_test, y_pred_lr)

print('='*45)
print('    HASIL EVALUASI LOGISTIC REGRESSION')
print('='*45)
print(f'  Akurasi: {acc_lr*100:.2f}%')
print()
print(classification_report(y_test, y_pred_lr,
      target_names=['negatif','netral','positif']))

## 💾 8. Simpan Model (untuk Streamlit App)

In [ ]:
import os
os.makedirs('model', exist_ok=True)

joblib.dump(nb,    'model/nb_model.pkl')
joblib.dump(lr,    'model/lr_model.pkl')
joblib.dump(tfidf, 'model/tfidf_vectorizer.pkl')

print('✅ Model berhasil disimpan:')
print('   model/nb_model.pkl')
print('   model/lr_model.pkl')
print('   model/tfidf_vectorizer.pkl')
print()
print('📌 File ini akan digunakan oleh app.py (Streamlit)')

## 📊 9. Visualisasi Lengkap

In [ ]:
labels = ['negatif', 'netral', 'positif']
colors = {'negatif':'#e74c3c', 'netral':'#95a5a6', 'positif':'#2ecc71'}
dist   = df['sentimen'].value_counts()
vals   = [dist.get(l,0) for l in labels]
f1_nb  = f1_score(y_test, y_pred_nb, average=None, labels=labels)
f1_lr  = f1_score(y_test, y_pred_lr, average=None, labels=labels)

EXTRA_SW = {
    'quot','amp','apos','nbsp','lt','gt','url','http','https',
    'tdk','yg','jg','sy','tp','ny','bgt','krn','sdh','klo','kl',
    'dr','pd','dgn','utk','dll','dsb','dst',
    'mbg','makan','bergizi','gratis',
    'prabowo','jokowi','joko','widodo','gibran','megawati',
    'anies','ganjar','ahok','sby','yudhoyono',
    'indonesia','jakarta','jawa','surabaya','bandung','medan',
    'tempo','kompas','detik','tribun','cnbc','okezone',
    'liputan','republika','kumparan','antara',
    'negara','rakyat','program','proyek','pemerintah',
    'presiden','menteri','anak','sekolah','siswa',
    'daerah','dinas','kabupaten','kota','provinsi',
    'anggaran','dana','uang','biaya','rupiah',
    'yang','dan','ini','itu','dari','dalam','akan','pada','juga',
    'saja','bisa','ada','hanya','jadi','sudah','atau',
}

fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('#f8f9fa')

# --- 1. Bar Distribusi ---
ax1 = fig.add_subplot(4,3,1)
bars = ax1.bar(labels, vals, color=[colors[l] for l in labels],
               edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, vals):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
             str(val), ha='center', fontweight='bold', fontsize=11)
ax1.set_title('Distribusi Sentimen', fontweight='bold', fontsize=12)
ax1.set_ylabel('Jumlah Komentar')
ax1.set_facecolor('#f8f9fa')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# --- 2. Pie Chart ---
ax2 = fig.add_subplot(4,3,2)
ax2.pie(vals, labels=labels, colors=[colors[l] for l in labels],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':2})
ax2.set_title('Proporsi Sentimen', fontweight='bold', fontsize=12)

# --- 3. Perbandingan Akurasi ---
ax3 = fig.add_subplot(4,3,3)
bars3 = ax3.bar(['Naïve Bayes','Logistic\nRegression'],
                [acc_nb*100, acc_lr*100],
                color=['#3498db','#e67e22'], edgecolor='white', linewidth=1.5, width=0.5)
for bar, acc in zip(bars3, [acc_nb*100, acc_lr*100]):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=12)
ax3.set_ylim(0,100)
ax3.set_title('Perbandingan Akurasi', fontweight='bold', fontsize=12)
ax3.set_ylabel('Akurasi (%)')
ax3.set_facecolor('#f8f9fa')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# --- 4. Confusion Matrix NB ---
ax4 = fig.add_subplot(4,3,4)
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=labels)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=labels, yticklabels=labels, cbar=False, linewidths=0.5)
ax4.set_title('Confusion Matrix\nNaïve Bayes', fontweight='bold', fontsize=12)
ax4.set_xlabel('Prediksi'); ax4.set_ylabel('Aktual')

# --- 5. Confusion Matrix LR ---
ax5 = fig.add_subplot(4,3,5)
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=labels)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=ax5,
            xticklabels=labels, yticklabels=labels, cbar=False, linewidths=0.5)
ax5.set_title('Confusion Matrix\nLogistic Regression', fontweight='bold', fontsize=12)
ax5.set_xlabel('Prediksi'); ax5.set_ylabel('Aktual')

# --- 6. F1-Score ---
ax6 = fig.add_subplot(4,3,6)
x = np.arange(len(labels))
ax6.bar(x-0.175, f1_nb, 0.35, label='Naïve Bayes', color='#3498db', edgecolor='white')
ax6.bar(x+0.175, f1_lr, 0.35, label='Logistic Regression', color='#e67e22', edgecolor='white')
ax6.set_xticks(x); ax6.set_xticklabels(labels)
ax6.set_ylim(0,1.0)
ax6.set_title('F1-Score per Kelas', fontweight='bold', fontsize=12)
ax6.set_ylabel('F1-Score')
ax6.legend(fontsize=9)
ax6.set_facecolor('#f8f9fa')
ax6.spines['top'].set_visible(False); ax6.spines['right'].set_visible(False)

# --- 7-9. Word Cloud ---
for idx, sentiment in enumerate(labels):
    ax = fig.add_subplot(4,3,7+idx)
    raw = ' '.join(df[df['sentimen']==sentiment]['processed'].dropna())
    filtered = ' '.join([w for w in raw.split()
                         if w not in EXTRA_SW and len(w) > 2])
    if filtered.strip():
        cmap = 'Greens' if sentiment=='positif' else 'Reds' if sentiment=='negatif' else 'Blues'
        wc = WordCloud(width=700, height=350, background_color='white',
                       max_words=80, colormap=cmap, prefer_horizontal=0.8,
                       stopwords=EXTRA_SW).generate(filtered)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud: {sentiment.upper()}', fontweight='bold', fontsize=12)

# --- 10. Top 10 Kata Negatif ---
ax10 = fig.add_subplot(4,3,10)
neg_w = [w for w in ' '.join(df[df['sentimen']=='negatif']['processed'].dropna()).split()
         if w not in EXTRA_SW and len(w) > 2]
fn = Counter(neg_w).most_common(10)
wn, cn = zip(*fn)
ax10.barh(list(wn)[::-1], list(cn)[::-1], color='#e74c3c', edgecolor='white')
ax10.set_title('Top 10 Kata Negatif', fontweight='bold', fontsize=12)
ax10.set_xlabel('Frekuensi')
ax10.set_facecolor('#f8f9fa')
ax10.spines['top'].set_visible(False); ax10.spines['right'].set_visible(False)

# --- 11. Top 10 Kata Positif ---
ax11 = fig.add_subplot(4,3,11)
pos_w = [w for w in ' '.join(df[df['sentimen']=='positif']['processed'].dropna()).split()
         if w not in EXTRA_SW and len(w) > 2]
fp = Counter(pos_w).most_common(10)
wp, cp = zip(*fp)
ax11.barh(list(wp)[::-1], list(cp)[::-1], color='#2ecc71', edgecolor='white')
ax11.set_title('Top 10 Kata Positif', fontweight='bold', fontsize=12)
ax11.set_xlabel('Frekuensi')
ax11.set_facecolor('#f8f9fa')
ax11.spines['top'].set_visible(False); ax11.spines['right'].set_visible(False)

# --- 12. Tabel Ringkasan ---
ax12 = fig.add_subplot(4,3,12)
ax12.axis('off')
metrics_data = [
    ['Akurasi',       f'{acc_nb*100:.2f}%',  f'{acc_lr*100:.2f}%'],
    ['F1 Negatif',    f'{f1_nb[0]:.3f}',      f'{f1_lr[0]:.3f}'],
    ['F1 Netral',     f'{f1_nb[1]:.3f}',      f'{f1_lr[1]:.3f}'],
    ['F1 Positif',    f'{f1_nb[2]:.3f}',      f'{f1_lr[2]:.3f}'],
    ['Data Train',    str(len(X_train)),        str(len(X_train))],
    ['Data Test',     str(len(X_test)),         str(len(X_test))],
    ['N-gram',        'Bigram (1,2)',           'Bigram (1,2)'],
    ['Stemming',      'Sastrawi ✓',            'Sastrawi ✓'],
]
table = ax12.table(cellText=metrics_data,
                   colLabels=['Metrik','Naïve Bayes','Logistic Reg.'],
                   cellLoc='center', loc='center',
                   colColours=['#bdc3c7','#3498db','#e67e22'])
table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1,1.5)
ax12.set_title('Ringkasan Evaluasi', fontweight='bold', fontsize=12)

plt.suptitle('Analisis Sentimen Komentar Program MBG\n(Naïve Bayes vs Logistic Regression — Pipeline Lengkap + Stemming)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('visualisasi_mbg.png', dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print('✅ Visualisasi disimpan: visualisasi_mbg.png')

## 🏆 10. Kesimpulan Akhir

In [ ]:
winner = 'Logistic Regression' if acc_lr > acc_nb else 'Naïve Bayes'
diff   = abs(acc_lr - acc_nb) * 100

print('=' * 60)
print('               KESIMPULAN AKHIR')
print('=' * 60)
print(f'\n📊 Total Data         : {len(df):,} komentar')
print(f'📊 Data Training      : {len(X_train):,} ({len(X_train)/len(df)*100:.0f}%)')
print(f'📊 Data Testing       : {len(X_test):,} ({len(X_test)/len(df)*100:.0f}%)')
print()
print(f'🤖 Akurasi Naïve Bayes        : {acc_nb*100:.2f}%')
print(f'🤖 Akurasi Logistic Regression: {acc_lr*100:.2f}%')
print()
print(f'🏆 Model Terbaik  : {winner}')
print(f'   Selisih akurasi: {diff:.2f}%')
print()
print('📌 Distribusi Sentimen:')
for s, c in df['sentimen'].value_counts().items():
    bar = '█' * (c // 100)
    print(f'   {s:10s}: {c:4d} ({c/len(df)*100:.1f}%) {bar}')
print()
print('📝 Perbaikan yang Dilakukan (v2):')
print('   ✅ Stemming Sastrawi ditambahkan ke pipeline preprocessing')
print('   ✅ Kamus lexicon diperbaiki: hapus kata ambigu (gratis, gizi, dll)')
print('   ✅ Confidence score ditambahkan untuk setiap label')
print('   ✅ Eksperimen unigram vs bigram — bigram terbukti lebih baik/setara')
print('   ✅ Model & vectorizer disimpan ke folder model/ untuk Streamlit app')